# 07 · Match explorer

Milestone 5's curve had one point per stat snapshot (minutes 3, 6, 9, 12, 15, then every 5). The explorer (`viz.explorer`, the dashboard's **Match lookup**) puts everything about a match on one time axis: a win probability that moves when things happen, what moved it, every player's souls and level, and every kill, death, item and buff.

Three questions before trusting it:
1. **Are the new curve points honest?** Between snapshots, some inputs are exact (buildings, mid-boss, kills) and some are estimated (souls, levels, damage). At snapshot times the rows must equal the training rows exactly.
2. **Do events move the curve the right way?** The first version made early buildings *lower* the scoring team's chances. Why, and the fix.
3. **What is a teamfight?** Grouping deaths into fights needs thresholds; we pick them from the data.

Then two matches walked through: a stomp and a comeback.

The 30 matches below come from the newest local partition. Their API metadata is cached in `data/matches/`, so re-running doesn't hit the API.

In [ ]:
import warnings
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.io as pio

from deadlock_predictor.data import match_detail as md
from deadlock_predictor.data.match import api_rows
from deadlock_predictor.features.snapshots import FEATURES
from deadlock_predictor.viz.curve import load_models, match_curve, match_features
from deadlock_predictor.viz.explorer import load_match, map_figure, match_figure
from deadlock_predictor.viz.fights import fights
from deadlock_predictor.viz.timeline import SIGN, features_at, objective_events

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
pio.renderers.default = "notebook_connected"
plt.rcParams["figure.figsize"] = (9, 4)
warnings.filterwarnings("ignore")
models = load_models()

# 30 ranked matches from the newest local partition, fetched once and cached in data/matches/.
raw = (ROOT / "data/raw/match_player/partition=123.parquet").as_posix()
ids = [
    r[0]
    for r in duckdb.sql(f"""
    SELECT DISTINCT match_id FROM '{raw}'
    WHERE match_mode = 'Ranked' AND game_mode = 'Normal'
    ORDER BY hash(match_id) LIMIT 30
""").fetchall()
]
matches = {}
for mid in ids:
    info = md.fetch_detail(mid)
    rows = api_rows(info)
    snaps, heroes, failed = match_features(rows)
    if not failed and len(snaps):
        matches[mid] = dict(
            info=info,
            rows=rows,
            snaps=snaps,
            heroes=heroes,
            deaths=md.deaths(info),
            stats=md.player_stats(info),
        )
print(f"{len(matches)} of {len(ids)} matches pass the training filters")

## 1 · What the match metadata holds

The dump keeps 10 stats per player snapshot. The API's match metadata (`data.match_detail`) has much more: every death with its second, killer and position; every item purchase; permanent buff pickups; ~50 stats per snapshot; and each player's position and health every second. For one match:

In [ ]:
info = matches[ids[0]]["info"] if ids[0] in matches else next(iter(matches.values()))["info"]
summary = {
    "stat snapshots per player": len(info["players"][0]["stats"]),
    "numbers per stat snapshot": sum(
        isinstance(v, int | float) for v in info["players"][0]["stats"][0].values()
    ),
    "deaths": len(md.deaths(info)),
    "item purchases": len(md.items(info)),
    "buff pickups": len(md.buffs(info)),
    "position samples": len(md.paths(info)),
    "buildings + mid-boss": len(objective_events(api_rows(info))),
}
pd.Series(summary)

## 2 · Event points: exact at snapshots

`viz.timeline.features_at` builds the model's input row at any second t:
- **buildings and mid-boss:** exact, counting events with 0 < time ≤ t, the same rule as training;
- **kills:** the snapshot's kills plus the enemy kills since, from the death list (the stats' own kill count and the death list disagree slightly, so this keeps snapshots exact);
- **souls, levels, assists, damage, farm:** linear between the surrounding snapshots, except building souls (below);
- **rank:** constant.

At the snapshot times these rows should equal the rows the model was trained on, exactly:

In [ ]:
worst = 0.0
for m in matches.values():
    at = features_at(m["snaps"], m["rows"], m["deaths"], m["snaps"].t, m["stats"])
    gap = np.abs(
        at[FEATURES].to_numpy(float) - m["snaps"].sort_values("t")[FEATURES].to_numpy(float)
    )
    worst = max(worst, gap.max())
print(f"largest difference from the training rows, over {len(matches)} matches: {worst}")

## 3 · Buildings and their souls

Each event gets two curve points: one second before and at the event. So the jump between them is that event's effect. Does it point toward the team that scored? Two versions of the souls between snapshots:
- **spread evenly:** souls interpolated linearly, so a building's soul bounty is smeared over the 3-5 minutes around it;
- **bounty at the event:** each snapshot interval's rise in a team's *objective souls* (`gold_boss` in the stats) is split over that team's buildings and mid-boss kills in the interval, and added at their exact second.

"Right direction" is the share of events that moved P toward the scorer; "mean move" is the average change in P toward the scorer.

In [ ]:
rows_out = []
for m in matches.values():
    ev = objective_events(m["rows"])
    ev = ev[(ev.t >= m["snaps"].t.min()) & (ev.t < m["info"]["duration_s"])].sort_values("t")
    if ev.empty:
        continue
    for souls in ("spread evenly", "bounty at the event"):
        stats = m["stats"] if souls == "bounty at the event" else None
        before = features_at(m["snaps"], m["rows"], m["deaths"], ev.t - 1, stats)
        after = features_at(m["snaps"], m["rows"], m["deaths"], ev.t, stats)
        p0 = match_curve(before, m["heroes"], {"final": models["final"]}).p_final.to_numpy()
        p1 = match_curve(after, m["heroes"], {"final": models["final"]}).p_final.to_numpy()
        toward = (p1 - p0) * ev.team.map(SIGN).to_numpy()
        rows_out += [
            dict(souls=souls, event=k, toward_scorer=d)
            for k, d in zip(ev.kind, toward, strict=True)
        ]
moves = pd.DataFrame(rows_out)
table = (
    moves.groupby(["event", "souls"])
    .toward_scorer.agg(events="size", right_direction=lambda x: (x > 0).mean(), mean_move="mean")
    .unstack("souls")
)
order = ["Lane Guardian", "Walker", "Base Guardian", "Shrine", "Patron Downed", "Mid-boss"]
table.reindex(order).round(3)

In [ ]:
moves.groupby("souls").toward_scorer.agg(
    right_direction=lambda x: (x > 0).mean(), mean_move="mean"
).round(3)

- **Spread evenly, early buildings point the wrong way:** a Lane Guardian lowered the scoring team's chances 94% of the time (6% right), and Walkers were barely better than a coin flip.
- **Why:** the model learned a building's effect *given* the souls lead, because in every training row the building's bounty is already part of `net_worth_diff`. "One more Guardian, same souls" means the team is doing worse elsewhere, and the model says so. Souls are a **mediator**: the building helps partly *through* the souls it gives. If you hold the mediator at the wrong value, you get the wrong sign. That's the same trap as "controlling for" something that sits on the causal path.
- **With the bounty at the event:** the right direction goes from 63% to 93%, and the average move toward the scorer quadruples (0.005 to 0.020). Late buildings (Base Guardians, Shrines, Patron) were already right: their effect on the model doesn't run through souls as much.
- The remaining wrong-direction Lane Guardians (21%) are small moves: early, the model sees a Guardian's value as mostly its souls.

## 4 · Teamfights

`viz.fights` groups deaths: a death joins an open fight if it's within `gap_s` of that fight's last death and within `radius` world units of its centre, otherwise it opens a new one. Several fights can be open at once, so fights in different lanes stay apart. Groups with fewer than 3 deaths are pick-offs, not fights.

First, how far apart are consecutive deaths?

In [ ]:
all_deaths = pd.concat([m["deaths"].assign(match=mid) for mid, m in matches.items()])
gaps = all_deaths.sort_values(["match", "t"]).groupby("match").t.diff().dropna()
fig, ax = plt.subplots()
ax.hist(gaps.clip(upper=180), bins=60)
ax.axvline(20, color="red", ls="--", label="fight_gap_s = 20")
ax.set(
    title="Seconds between consecutive deaths (any team)",
    xlabel="seconds (clipped at 180)",
    ylabel="count",
)
ax.legend()
print(f"median gap {gaps.median():.0f} s; share under 20 s: {(gaps < 20).mean():.0%}")

In [ ]:
grid = []
for gap_s in (10, 20, 30):
    for radius in (1500, 3000, 6000):
        found = [fights(m["deaths"], gap_s, radius, 3) for m in matches.values()]
        n = sum(len(f) for f in found)
        in_fights = sum((f.deaths_team0 + f.deaths_team1).sum() for f in found)
        grid.append(
            dict(
                gap_s=gap_s,
                radius=radius,
                fights_per_match=n / len(matches),
                share_of_deaths=in_fights / len(all_deaths),
            )
        )
pd.DataFrame(grid).pivot(index="gap_s", columns="radius").round(2)

In [ ]:
# Do fights move the curve toward the side that won them?
out = []
for m in matches.values():
    f = fights(m["deaths"], 20, 3000, 3)
    f = f[(f.start - 1 >= m["snaps"].t.min()) & (f.end < m["info"]["duration_s"])]
    f = f[f.winner.notna()]
    if f.empty:
        continue
    for t0, t1, w, k in zip(
        f.start - 1, f.end, f.winner, f.deaths_team0 + f.deaths_team1, strict=True
    ):
        at = features_at(m["snaps"], m["rows"], m["deaths"], [t0, t1], m["stats"])
        p = match_curve(at, m["heroes"], {"final": models["final"]}).p_final.to_numpy()
        out.append(dict(size=k, toward_winner=(p[1] - p[0]) * SIGN[w]))
fm = pd.DataFrame(out)
fm.groupby(fm["size"].clip(upper=6)).toward_winner.agg(
    fights="size", right_direction=lambda x: (x > 0).mean(), mean_move="mean"
).round(3)

- **Half of all deaths come within 14 s of the previous one**, so deaths cluster. 20 s sits on the shoulder of the distribution: long enough to keep a fight together, short enough not to chain separate ones.
- **The grid:** a wider radius mostly merges fights in neighbouring lanes. The defaults `gap_s = 20`, `radius = 3000` (the map is about 20,000 units across) give ~8 fights per match covering 39% of deaths, which matches the feel of a Deadlock match: a handful of real fights and many pick-offs.
- **Fights barely move the curve, at the time.** Fights of 4+ deaths move P toward the winner 74-92% of the time, but by only ~0.02. That's not because fights don't matter. The model values kills mostly through the **souls** they give, and between snapshots souls are interpolated, so a fight's soul swing reaches the curve gradually or at the next snapshot. The same fix as for buildings should work: player-kill souls (`gold_player` in the stats) stepped in at each kill. That's a good next improvement.

## 5 · Two matches

Among the 30 matches, the **stomp** is the one whose loser never got above the lowest peak, and the **comeback** is the one whose loser came closest to winning. Hover the figures for details (they render when the notebook runs; outputs are stripped from the repository).

In [ ]:
def swing(mid):
    m = load_match(mid, models)
    p = m.points.p_final
    loser_best = p.max() if m.winner == "Team1" else 1 - p.min()
    return m, loser_best


found = {mid: swing(mid) for mid in list(matches)[:30]}
stomp = min(found, key=lambda mid: found[mid][1])
comeback = max(found, key=lambda mid: found[mid][1])
print(
    f"stomp: {stomp} (loser never above {found[stomp][1]:.0%}); "
    f"comeback: {comeback} (loser reached {found[comeback][1]:.0%})"
)

### The stomp

The winner leads from the start and never lets go: the probability climbs through the early fights and every building after. Most of the per-player rows show who carried (kills) and who was on the receiving end (deaths, with grey time-dead bars).

In [ ]:
match_figure(found[stomp][0]).show()

### The comeback

The Archmother reached 98%, and the Hidden King won anyway:
- **0-30 min:** the Hidden King starts at 24%, almost all from the draft (the hero bar at minute 3), and falls to 2% by minutes 25-30, 18k souls behind. Both of those are exact snapshot points, not interpolation.
- **32 min:** a 4-1 fight for the Hidden King. The curve hardly moves at the time (section 4's limitation), but the souls lead swings over the next snapshot.
- **40 min:** another 3-0. The souls gap is closed and P is back to ~20%.
- **44-46 min:** the mid-boss, then a 5-0 fight, and the Hidden King runs through two Base Guardians, both Shrines and the Patron.

Is 98% → loss a sign the model is wrong? Not by itself: a calibrated 2% should still happen 1 time in 50. Notebook 05 checked exactly this on the test split: teams written off at ≤10% won 5.2% of the time, against 5.4% expected.

In [ ]:
m = found[comeback][0]
match_figure(m).show()

In [ ]:
# Where everyone was during the biggest fight of the comeback.
if len(m.fights):
    big = (
        m.fights.assign(size=m.fights.deaths_team0 + m.fights.deaths_team1)
        .sort_values("size")
        .iloc[-1]
    )
    map_figure(m, (big.start + big.end) / 2).show()

The map at the middle of the biggest fight: dots are players (size = health), lines are the last 30 s of movement, ✕ are deaths around that moment. The faint background is every position anyone held during the match, which draws the lanes and bases without needing the map image.

## Summary

- **The explorer** puts the whole match on one time axis: win probability at every snapshot, building, mid-boss and fight; what moved it; souls and levels per player; kills, deaths, items and buffs per player; and a map.
- **Event points are exact at snapshots** (difference 0.0 on 29 matches) and estimated in between. Souls between snapshots use the *next* snapshot, so in-between points are the best estimate of the state at that moment, not what a live viewer could know.
- **Mediators:** a model's effect for one feature is *given the others*. Feeding it a building without the souls that come with it reversed the sign. Adding the bounty at the event fixed the direction (63% → 93%).
- **Fights:** deaths within 20 s and 3,000 units form a fight (~8 per match, 39% of deaths). Their effect reaches the curve mostly through souls, so stepping kill souls in at each kill is the natural next step.
- Run it: `streamlit run src/deadlock_predictor/app/dashboard.py`, then **Match lookup**. Matches deadlock-api has already processed load instantly. Newer ones are fetched from Steam, which is rate-limited (about 3 per hour).